# TinyTetris demo

TinyTetris runs on tinycpu. Jupyter only writes input and displays app status plus the framebuffer mirror.

In [ ]:
from pynq import Overlay, MMIO
from tinycpu_loader import TinyCPULoader, find_loader_ip
from tetris_controller import TetrisController

overlay = Overlay("tinycpu.bit")
overlay.ip_dict

In [ ]:
LOADER_IP_NAME = None
ip = find_loader_ip(overlay, preferred_name=LOADER_IP_NAME)
cpu = TinyCPULoader(MMIO(ip["phys_addr"], ip["addr_range"]))
tetris = TetrisController(cpu)

cpu.reset()
cpu.load_hex("../programs/tetris/firmware.hex")
cpu.set_boot_pc(0)
cpu.run()
tetris.start()

In [ ]:
import ipywidgets as widgets
from IPython.display import display

buttons = [
    ("Left", tetris.left),
    ("Right", tetris.right),
    ("Rotate", tetris.rotate),
    ("Soft Drop", tetris.soft_drop),
    ("Hard Drop", tetris.hard_drop),
    ("Hold", tetris.hold),
    ("Start", tetris.start),
    ("Pause", tetris.pause),
]

widgets_list = []
for label, callback in buttons:
    button = widgets.Button(description=label)
    button.on_click(lambda _, cb=callback: cb())
    widgets_list.append(button)

display(widgets.HBox(widgets_list))


In [ ]:
from IPython.display import clear_output
import time

PIECE_NAMES = [".", "I", "J", "L", "O", "S", "T", "Z"]

for _ in range(200):
    cells = cpu.read_framebuffer()
    clear_output(wait=True)
    cpu.print_board(cells)
    next_piece = tetris.next_piece()
    held_piece = tetris.held_piece()
    next_name = PIECE_NAMES[next_piece] if next_piece < len(PIECE_NAMES) else "?"
    hold_name = PIECE_NAMES[held_piece] if held_piece < len(PIECE_NAMES) else "?"
    print("status:", hex(cpu.read_app_status()))
    print("score:", tetris.score(), "lines:", tetris.lines(), "level:", tetris.level())
    print("next:", next_name, "hold:", hold_name)
    print("frame:", cpu.read_frame_counter())
    time.sleep(0.1)
